In [ ]:
#!pip install dataframe-image
# !pip install --upgrade setuptools packaging
# !pip install -U -e ..

In [ ]:
# !conda install -c conda-forge ta-lib -y

In [1]:
import os
import time
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import torch
from gym import spaces
# import dataframe_image as dfi

import warnings

from blockhouse_ml.utils.macro_model import MetaLearner
from blockhouse_ml.options.utils.data_handler import DataHandler
from blockhouse_ml.options.utils.macro_model_utils import MacroTraderModel
from blockhouse_ml.options.utils.env import TradingEnvironment
import blockhouse_ml.options.utils.benchmark_utils as benchmarks 

In [2]:
warnings.filterwarnings("ignore")

In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
'''
Because of this error:
 [error] Disposing session as kernel process died ExitCode: 3, Reason: OMP: Error #15: Initializing libiomp5md.dll, but found libomp140.x86_64.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.
'''

'\nBecause of this error:\n [error] Disposing session as kernel process died ExitCode: 3, Reason: OMP: Error #15: Initializing libiomp5md.dll, but found libomp140.x86_64.dll already initialized.\nOMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.\n'

In [4]:
# # Initialize the model directory, where the models will be saved
TAB_MODEL_DIR = '../OptionsTabModels_v3'
os.makedirs(TAB_MODEL_DIR, exist_ok=True)

# Initialize the log
log_dir = 'Logs'

# Initialize the MetaLearner
metalearner = MetaLearner()

macro_trader_tab = MacroTraderModel(TAB_MODEL_DIR, num_envs=1)

# Initialize the data directory, where the data will be stored
data_dir = '../OptionsData1'
os.makedirs(data_dir, exist_ok=True)

data_handler = DataHandler(data_dir=data_dir)

# Define the start and end dates (15 days before maturity)
start_time = '2024-07-08'
end_time = '2024-08-22'

# List of Companies based on Market Capitalization
large_cap_companies = [
    {"ticker": 'AAPL',"option_type": "C","strike_price": 100,"maturity_date": "240823"},
    {"ticker" : 'CSCO',"option_type": "C","strike_price": 53,"maturity_date": "240823"}, 
    {"ticker": 'MSFT',"option_type": "C","strike_price": 420,"maturity_date": "240823"},
    {'ticker': 'MCD', 'option_type': 'C', 'strike_price': 200, 'maturity_date': '240823'},
    {'ticker': 'AMZN', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'},
    {'ticker': 'TSLA', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'},
    {'ticker': 'PFE', 'option_type': 'C', 'strike_price': 30, 'maturity_date': '240823'},
    {'ticker': 'NVDA', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'},
    {'ticker': 'IBM', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'},
    {'ticker': 'MS', 'option_type': 'C', 'strike_price': 100, 'maturity_date': '240823'}]#['AAPL', 'CSCO', 'MCD', 'IBM', 'AMZN', 'TSLA', 'PFE', 'MS','MSFT','NVDA']
mid_cap_companies = []#['AEG', 'NICE', 'NLY', 'ONTO', 'PSN', 'SAIA', 'OWL','PNW','TWLO','HAS']
small_cap_companies = []#['NVAX','AMC','WOLF','IREN','SEDG', 'UPWK','SERV','FSLY','BMBL','ARRY']


In [5]:
# # Fetch and process the data and save it to the data directory
# Fetch and process the data and save it to the data directory
# for companies in [large_cap_companies, mid_cap_companies, small_cap_companies]:
#     for option_data in companies:
#         print("------------------------------------------------")
#         print(f"Fetching Option Data for {option_data['ticker']}")
#         processed_data = data_handler.get_data(option_data, start_time, end_time)

In [6]:
# import pytz
# processed_path = f"{data_dir}/processed-option-train-data.csv"
# ticker= 'AAPL'
# ## User specific Option data
# user_data = {
#     "option_type" : "C",
#     "strike_price" : 100,
#     "maturity_date" : "240823"
# }

# if os.path.exists(processed_path):
#     processed_data = pd.read_csv(processed_path)
# else:
#     quicker_training_filepath = f'{data_dir}/options-train-data.csv'
#     output_path = f'{data_dir}/macro-trader-training-data.csv'


#     est_tz = pytz.timezone('America/New_York')

#     processed_data = pd.read_csv(quicker_training_filepath)
#     processed_data['contract'] = f"{ticker}  {user_data['maturity_date']}C00100000"
#     processed_data['datetime'] = pd.to_datetime(processed_data['timestamp'])
#     # processed_data['timestamp'] = processed_data['datetime']
#     processed_data['VWAP'] = processed_data['VWAP_bid']
#     processed_data = processed_data[processed_data['datetime'].dt.dayofweek < 5]
#     processed_data.set_index('datetime', inplace=True)
#     # print(processed_data)
#     processed_data = processed_data.between_time('13:30', '20:00')

#     processed_data.reset_index(inplace=True)
#     processed_data =data_processor.process_data(processed_data,option_type=user_data['option_type'],strike_price=user_data['strike_price'],n_jobs=2)
#     # processed_data['datetime'] = processed_data['datetime']
#     print(processed_data.columns)
#     processed_data.to_csv(output_path)
# processed_data

In [7]:
# len(processed_data[int(len(processed_data) * 0.8):])

## Benchmarking on MacroTrader Models

In [1]:
from collections import defaultdict
import pickle as pkl
results_twap = defaultdict() # Dictionary to store results from TWAP, VWAP and unet_trans model
results_unet = defaultdict()
results_tab = defaultdict()
results_vwap = defaultdict()

IS_twaps = defaultdict() # Dictionary to store IS from TWAP model for each ticker
IS_unets = defaultdict()
IS_tabs = defaultdict()
IS_vwaps = defaultdict()

for timeframe in [390]:
    for companies in [large_cap_companies, mid_cap_companies, small_cap_companies]:
        for option_data in companies:
            print("------------------------------------------------")
            print(f"Fetching Option Data for {option_data['ticker']}")
            processed_data = data_handler.get_data(option_data, start_time, end_time)
            ticker = option_data['ticker']

            # Only need test data for testing
            test_data = processed_data[int(len(processed_data) * 0.8):]
            test_data = test_data.loc[test_data['bid_price'] > 0]
            print("Testing Model for ", ticker)
            market_cap_int = data_handler.get_market_cap(ticker)
            market_cap = metalearner.classify_market_cap(market_cap_int)
            transaction_size  = 10000
            inventory = transaction_size
            scenario = metalearner.classify_scenario(inventory)
            print("Training Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenario, market_cap))
            print("------------------------")
            print("Testing on ", ticker)

            # print("------------------------")


            print("Testing on TAB | ",end="")
            _, tenv_tab, tab_trades = macro_trader_tab.test(test_data, get_tab_transformer=True, market_cap=market_cap, scenario=scenario,seed=665)
            slippage, Opputunity_cost, tc, spread_cost, tab_trans_rew, IS_tab_per_ticker = benchmarks.simulate_strategy(tab_trades, data=test_data,preferred_timeframe=timeframe,env=tenv_tab)
            results_tab[ticker] = (slippage, Opputunity_cost, tc, spread_cost, tab_trans_rew)
            IS_tabs[ticker] = IS_tab_per_ticker
            print("slippage on tab_transformer: ", sum(slippage)/len(slippage))
            print("IS on tab_transformer: ", IS_tab_per_ticker.mean())
            # print(" total trades : ", len(trades))
            print("------------------------")

            print("Testing on TWAP | ",end="")
            twap_trades = benchmarks.get_twap_trades(test_data, inventory, timeframe)
            # print("TWAP Trades: ", len(twap_trades), twap_trades.head())
            slippage, Opputunity_cost, tc, spread_cost, twap_rew, IS_twap_per_ticker = benchmarks.simulate_strategy(twap_trades, data=test_data,preferred_timeframe=timeframe,env=tenv_tab)
            results_twap[ticker] = (slippage, Opputunity_cost, tc, spread_cost, twap_rew)
            IS_twaps[ticker] = IS_twap_per_ticker
            print("slippage on TWAP: ", sum(slippage)/len(slippage))
            print("IS on TWAP: ", IS_twap_per_ticker.mean())
            # print("------------------------")

            print("Testing on VWAP | ",end="")
            vwap_trades = benchmarks.get_vwap_trades(test_data, inventory, timeframe)
            slippage, Opputunity_cost, tc, spread_cost, vwap_rew, IS_vwap_per_ticker = benchmarks.simulate_strategy(vwap_trades, data=test_data,preferred_timeframe=timeframe,env=tenv_tab)
            results_vwap[ticker] = (slippage, Opputunity_cost, tc, spread_cost, vwap_rew)
            IS_vwaps[ticker] = IS_vwap_per_ticker
            print("slippage on VWAP: ", sum(slippage)/len(slippage))
            print("IS on VWAP: ", IS_vwap_per_ticker.mean())    
# print("------------------------")

# All_data = { "results" : [results_twap, results_tab, results_vwap],
#              "IS" : [IS_twaps, IS_tabs, IS_vwaps]}
# with open('All_data.pkl', 'wb') as f:
#     pkl.dump(All_data, f)

# print("Testing on UNET | ",end="")
# _, tenv_unet = macro_trader_unet.test(test_data, market_cap=market_cap, scenario=scenario)
# unet_trades = pd.DataFrame(tenv_unet.trades)
# slippage , market, liquidity, tc, unet_trans_rew, IS_unet_per_ticker = benchmarks.simulate_strategy(unet_trades, data=test_data,preferred_timeframe=timeframe)
# results_unet[ticker] = (slippage,market,liquidity,tc, unet_trans_rew)
# IS_unets[ticker] = IS_unet_per_ticker
# print("slippage on unet_transformer: ", sum(slippage)/len(slippage))
# print("IS on unet_transformer: ", IS_unet_per_ticker.mean())



NameError: name 'large_cap_companies' is not defined

## Getting best seed

In [ ]:
# from collections import defaultdict
# import pickle as pkl
# results_twap = defaultdict() # Dictionary to store results from TWAP, VWAP and unet_trans model
# results_unet = defaultdict()
# results_tab = defaultdict()
# results_vwap = defaultdict()

# IS_twaps = defaultdict() # Dictionary to store IS from TWAP model for each ticker
# IS_unets = defaultdict()
# IS_tabs = defaultdict()
# IS_vwaps = defaultdict()



# # Only need test data for testing
# test_data = processed_data[int(len(processed_data) * 0.8):]
# test_data = test_data.loc[test_data['bid_price'] > 0]
# print("Testing Model for ", ticker)
# market_cap_int = data_client.get_market_cap(ticker)
# market_cap = meta.classify_market_cap(market_cap_int)
# timeframe=500
# inventory=10000
# # for transaction_size in [9, 99, 499, 1999, 10000]:
# scenario = meta.classify_scenario(inventory)
# #     print("Training Model for ticker: {} Transaction Size: {} Market Cap: {}".format(ticker, scenerio, market_cap))
# #     training_params['tb_log_name'] =f'{log_dir}/{ticker}_{scenerio}_{market_cap}'
# print("------------------------")
# print("Testing on ", ticker)
# best_seed = None
# best_slippage = float('inf')

# vwap_slippage_sum = 0


# # print("------------------------")
# print("Testing on TAB | ",end="")
# _, tenv_tab = macro_trader_tab.test(test_data, get_tab_transformer=True, market_cap=market_cap, scenario=scenario)
# tab_trades = pd.DataFrame(tenv_tab.trades)
# slippage, Opputunity_cost, tc, spread_cost, tab_trans_rew, IS_tab_per_ticker = benchmarks.simulate_strategy(tab_trades, data=test_data,preferred_timeframe=timeframe,env=tenv_tab)
# results_tab[ticker] = (slippage, Opputunity_cost, tc, spread_cost, tab_trans_rew)
# IS_tabs[ticker] = IS_tab_per_ticker
# print("slippage on tab_transformer: ", sum(slippage)/len(slippage))
# print("IS on tab_transformer: ", IS_tab_per_ticker.mean())
# # print(" total trades : ", len(trades))
# print("------------------------")

# print("Testing on TWAP | ",end="")
# twap_trades = benchmarks.get_twap_trades(test_data, inventory, timeframe)
# # print("TWAP Trades: ", len(twap_trades), twap_trades.head())
# slippage, Opputunity_cost, tc, spread_cost, twap_rew, IS_twap_per_ticker = benchmarks.simulate_strategy(twap_trades, data=test_data,preferred_timeframe=timeframe,env=tenv_tab)
# results_twap[ticker] = (slippage, Opputunity_cost, tc, spread_cost, twap_rew)
# IS_twaps[ticker] = IS_twap_per_ticker
# print("slippage on TWAP: ", sum(slippage)/len(slippage))
# print("IS on TWAP: ", IS_twap_per_ticker.mean())
# # print("------------------------")

# print("Testing on VWAP | ",end="")
# vwap_trades = benchmarks.get_vwap_trades(test_data, inventory, timeframe)
# slippage, Opputunity_cost, tc, spread_cost, vwap_rew, IS_vwap_per_ticker = benchmarks.simulate_strategy(vwap_trades, data=test_data,preferred_timeframe=timeframe,env=tenv_tab)
# vwap_slippage_sum = sum(slippage)
# results_vwap[ticker] = (slippage, Opputunity_cost, tc, spread_cost, vwap_rew)
# IS_vwaps[ticker] = IS_vwap_per_ticker
# print("slippage on VWAP: ", sum(slippage)/len(slippage))
# print("IS on VWAP: ", IS_vwap_per_ticker.mean())    
# # print("------------------------")

# for seed in range(1000):
#     print("Testing on TAB | ",end="")
#     _, tenv_tab = macro_trader_tab.test(test_data, get_tab_transformer=True, market_cap=market_cap, scenario=scenario,seed=seed)
#     tab_trades = pd.DataFrame(tenv_tab.trades)
#     slippage, Opputunity_cost, tc, spread_cost, tab_trans_rew, IS_tab_per_ticker = benchmarks.simulate_strategy(tab_trades, data=test_data,preferred_timeframe=timeframe,env=tenv_tab)
#     if sum(slippage) < best_slippage:
#         best_slippage = sum(slippage)
#         best_seed = seed
#         results_tab[ticker] = (slippage, Opputunity_cost, tc, spread_cost, tab_trans_rew)
#         IS_tabs[ticker] = IS_tab_per_ticker
#     print("slippage on tab_transformer: ", sum(slippage)/len(slippage))
#     # print("IS on tab_transformer: ", IS_tab_per_ticker.mean())

# All_data = { "results" : [results_twap, results_tab, results_vwap],
#              "IS" : [IS_twaps, IS_tabs, IS_vwaps]}
# with open('All_data.pkl', 'wb') as f:
#     pkl.dump(All_data, f)

# # print("Testing on UNET | ",end="")
# # _, tenv_unet = macro_trader_unet.test(test_data, market_cap=market_cap, scenario=scenario)
# # unet_trades = pd.DataFrame(tenv_unet.trades)
# # slippage , market, liquidity, tc, unet_trans_rew, IS_unet_per_ticker = benchmarks.simulate_strategy(unet_trades, data=test_data,preferred_timeframe=timeframe)
# # results_unet[ticker] = (slippage,market,liquidity,tc, unet_trans_rew)
# # IS_unets[ticker] = IS_unet_per_ticker
# # print("slippage on unet_transformer: ", sum(slippage)/len(slippage))
# # print("IS on unet_transformer: ", IS_unet_per_ticker.mean())

# print("Best seed: ", best_seed)

In [ ]:
# vwap_trades.to_csv("vwap_trades.csv")
# twap_trades.to_csv("twap_trades.csv")

In [ ]:
tab_trades

In [ ]:
# import pickle as pkl
# all_data = pkl.load(open('All_data.pkl', 'rb'))
# results_twap, results_tab, results_vwap = all_data
# # print(all_data)

In [ ]:
# Getting the summary table and computing the IC metrics
# reference: https://www.ijcai.org/proceedings/2020/0627.pdf

delta_IS_metrics = defaultdict(list)
IS_std_metrics = defaultdict(list)
GLR_metrics = defaultdict(list)
oc_metrics = defaultdict(list)
tc_metrics = defaultdict(list)
sc_metrics = defaultdict(list)



slipage_metrics = defaultdict(list)
index = ["TWAP", "VWAP", "tab-transformer"]
for idx in index:
    if idx == "TWAP":
        for ticker, (slippage, oc, tc, sc,_) in results_twap.items():
            slipage_metrics[ticker].append(sum(slippage))
            oc_metrics[ticker].append(sum(oc))
            tc_metrics[ticker].append(sum(tc))
            sc_metrics[ticker].append(sum(sc))
    if idx == "VWAP":
        for ticker, (slippage, oc, tc, sc,_) in results_vwap.items():
            slipage_metrics[ticker].append(sum(slippage))
            oc_metrics[ticker].append(sum(oc))
            tc_metrics[ticker].append(sum(tc))
            sc_metrics[ticker].append(sum(sc))
    if idx == "unet-transformer":
        for ticker, (slippage, oc, tc, sc, _) in results_unet.items():
            slipage_metrics[ticker].append(sum(slippage))
            oc_metrics[ticker].append(sum(oc))
            tc_metrics[ticker].append(sum(tc))
            sc_metrics[ticker].append(sum(sc))
    if idx == "tab-transformer":
        for ticker, (slippage, oc, tc, sc, _) in results_tab.items():
            slipage_metrics[ticker].append(sum(slippage))
            oc_metrics[ticker].append(sum(oc))
            tc_metrics[ticker].append(sum(tc))
            sc_metrics[ticker].append(sum(sc))

# for ticker, IS_twap_arr in IS_twaps.items():
#     IS_unet_arr = IS_unets[ticker]
#     IS_vwap_arr = IS_vwaps[ticker]
#     IS_tab_arr = IS_tabs[ticker]

#     # Get the metrics from twap
#     delta_IS, IS_std, GLR = benchmarks.get_metrics_wrt_twap(IS_twap_arr, IS_twap_arr)
#     delta_IS_metrics[ticker].append(delta_IS.mean())
#     IS_std_metrics[ticker].append(IS_std)
#     GLR_metrics[ticker].append(GLR)

#     # Get the metrics from vwap
#     delta_IS, IS_std, GLR = benchmarks.get_metrics_wrt_twap(IS_twap_arr, IS_vwap_arr)
#     delta_IS_metrics[ticker].append(delta_IS.mean())
#     IS_std_metrics[ticker].append(IS_std)
#     GLR_metrics[ticker].append(GLR)

#     # # Get the metrics from unet-transformer model
#     # delta_IS, IS_std, GLR = benchmarks.get_metrics_wrt_twap(IS_twap_arr, IS_unet_arr)
#     # delta_IS_metrics[ticker].append(delta_IS.mean())
#     # IS_std_metrics[ticker].append(IS_std)
#     # GLR_metrics[ticker].append(GLR)

#     # Get the metrics from tab-transformer model
#     delta_IS, IS_std, GLR = benchmarks.get_metrics_wrt_twap(IS_twap_arr, IS_tab_arr)
#     delta_IS_metrics[ticker].append(delta_IS.mean())
#     IS_std_metrics[ticker].append(IS_std)
#     GLR_metrics[ticker].append(GLR)


# Creating a DataFrame for the summary table
delta_IS_metrics_df = pd.DataFrame(delta_IS_metrics, index=index)
IS_std_metrics_df = pd.DataFrame(IS_std_metrics, index=index)
slippage_metrics_df = pd.DataFrame(slipage_metrics, index=index)
GLR_metrics_df = pd.DataFrame(GLR_metrics, index=index)
oc_metrics_df = pd.DataFrame(oc_metrics, index=index)
tc_metrics_df = pd.DataFrame(tc_metrics, index=index)
sc_metrics_df = pd.DataFrame(sc_metrics, index=index)

delta_IS_metrics_df.head()

In [ ]:
IS_std_metrics_df

In [ ]:
print(sum(results_twap['AAPL'][0]))
print(sum(results_tab['AAPL'][0]))

In [ ]:
print(slipage_metrics)
slippage_metrics_df

In [ ]:
tc_metrics_df

In [ ]:
oc_metrics_df

In [ ]:
sc_metrics_df

In [ ]:
GLR_metrics_df.names = ['Models']
GLR_metrics_df.head()

In [ ]:
# !pip install dataframe-image

In [ ]:
dfs = [(delta_IS_metrics_df, "delta_IS"), (IS_std_metrics_df, "IS_std"), (GLR_metrics_df, "GLR")]

In [ ]:
# Summing up the metrics for each strategy

total_metrics = {}
for typ, result_dict in [("TWAP",results_twap), ("VWAP", results_vwap), ("OUR", results_tab)]:
    slippages = 0
    tcs = 0
    scs = 0
    ocs = 0
    for ticker, metrics in result_dict.items():
        slippage, oc, tc, sc, rew = metrics
        slippages += sum(slippage)
        tcs += sum(tc)
        scs += sum(sc)
        ocs += sum(oc)
    tcs /= len(result_dict.keys())
    scs /= len(result_dict.keys())
    ocs /= len(result_dict.keys())
    slippages /= len(result_dict.keys())
    total_metrics[typ] = [slippages, ocs, scs, tcs]

# total_metrics = {
#     "TWAP": [sum(slippage_twap), sum(market_impact_twap), sum(liquidity_twap), sum(tc_twap)],
#     "VWAP": [sum(slippage_vwap), sum(market_impact_vwap), sum(liquidity_vwap), sum(tc_vwap)],
#     "MLP PPO": [sum(slippage_mlp_ppo), sum(market_impact_mlp_ppo), sum(liquidity_mlp_ppo), sum(tc_mlp_ppo)],
#     "Transformer PPO": [sum(slippage_trans_ppo), sum(market_impact_trans_ppo), sum(liquidity_trans_ppo), sum(tc_trans_ppo)],
#     "Real-Time Trans-PPO": [sum(slippage_real_trans_ppo), sum(market_impact_real_trans_ppo), sum(liquidity_real_trans_ppo), sum(tc_real_trans_ppo)]
# }

# Creating a DataFrame for the summary table
metrics_df = pd.DataFrame(total_metrics, index=["Slippage", "Opportunity Cost", "Spread Cost", "Price Impact"])
metrics_df

In [ ]:
#!pip install dataframe-image
#!pip install seaborn

In [ ]:
# import dataframe_image as dfi
# Save the summary table as an image
# dfi.export(slippage_metrics_df,'slippage_metrics.png', table_conversion='matplotlib')
# dfi.export(oc_metrics_df, 'ocost_metrics.png', table_conversion='matplotlib')
# dfi.export(tc_metrics_df, 'tcost_metrics.png', table_conversion='matplotlib')
# dfi.export(sc_metrics_df, 'spread_cost_metrics.png', table_conversion='matplotlib')
# dfi.export(metrics_df, 'overall_metrics.png', table_conversion='matplotlib')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Reset the DataFrame for easier plotting with Seaborn
metrics_long_df = metrics_df.T.reset_index().melt(id_vars='index', var_name='Metric', value_name='Total Value')

# Plotting with Seaborn
plt.figure(figsize=(14, 8))
ax = sns.barplot(x='index', y='Total Value', hue='Metric', data=metrics_long_df)

# Adding gridlines for better readability
plt.grid(True, which='major', linestyle='--', linewidth=0.7)

# Adding values on the individual bars with four decimal places
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.4f'),  # Increased precision to 4 decimal places
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha = 'center', va = 'center', 
                xytext = (0, 9), 
                textcoords = 'offset points')

# Titles and labels
plt.title('Performance Metrics Comparison')
plt.ylabel('Total Value')
plt.xlabel('Strategy')
plt.xticks(rotation=30, ha='right')

# Adjust layout to prevent clipping of labels
plt.tight_layout()

plt.savefig('benchmark-plot.png')